In [1]:
# pip install --upgrade google-cloud-bigquery
# pip install pydata-google-auth
# pip install db-dtypes

In [2]:
# Turn off warnings
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Authenticate through your browser with you Big Query account
import pydata_google_auth

scopes = ['https://www.googleapis.com/auth/bigquery']

credentials = pydata_google_auth.get_user_credentials(
    scopes,
    # Set auth_local_webserver to True for a more convenient flow,
    # but be aware it may not work on a remote server
    # like a cloud-hosted notebook if there are network restrictions
    # auth_local_webserver=True, # Optional
)

In [4]:
from google.cloud import bigquery

project = 'product-project-463105'

client = bigquery.Client(project=project,credentials=credentials)
google_citibike = '`bigquery-public-data.new_york_citibike.citibike_trips`'

# Question 2, Part A

In [5]:

# What was the busiest start station by trip count, and how many trips started there?
query = f'''
-- Count distinct trips and use qualify with rank to get the start_station_id with the most trips
SELECT start_station_id, start_station_name, COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) d_trips
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
GROUP BY 1,2
--ORDER BY 3 DESC

QUALIFY RANK() OVER(ORDER BY COUNT(*) DESC) = 1;

'''

busy_station_df = client.query(query).to_dataframe()
busy_station_df

,start_station_id,start_station_name,d_trips
0,519,Pershing Square North,135906


### Q: What was the busiest start station by trip count, and how many trips started there?
### A: The busiest station is Pershing Square North, adjacent to Grand Central. This makes sense as a commuter hub.<br>

In [20]:

# What was the median trip duration for Subscriber vs Customer? How do you read the gap?
query = f'''
-- median trip length by usertype
SELECT DISTINCT usertype, PERCENTILE_CONT(tripduration, 0.5) OVER(PARTITION BY usertype) AS median_tripduration
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
AND usertype IS NOT NULL;

'''

median_duration_df = client.query(query).to_dataframe()
# Convert seconds to minutes
median_duration_df['median_minutes'] = (median_duration_df['median_tripduration'] / 60).round(1)
median_duration_df[['usertype','median_minutes']]

,usertype,median_minutes
0,Customer,21.5
1,Subscriber,9.7


### Q: What was the median trip duration for Subscriber vs Customer? How do you read the gap?
### A: Customers ride at a median of ~21.5 minutes vs ~9.7 for Subscribers. The gap reflects different use cases.
### Subscribers are commuters doing short point-to-point trips, Customers are casual/tourist riders on longer leisure trips.<br>

In [31]:

# What share of trips were round trips (same start and end station)? 
query = f'''
-- Get the pct of roundtrip trips using distinct inside the count to make robust to dups
SELECT ROUND(COUNT(DISTINCT CASE WHEN start_station_id = end_station_id THEN bikeid || CAST(starttime AS STRING) END) / 
        COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) * 100.0,1) pct_roundtrip
FROM
`bigquery-public-data.new_york_citibike.citibike_trips` 

WHERE EXTRACT(YEAR FROM starttime) = 2017;

'''

pct_rndtrip_df = client.query(query).to_dataframe()
display(pct_rndtrip_df.style.format({'pct_roundtrip': '{:.0f}%'}))


# Does it differ by user type?
query = f'''
-- Adding usertype cut to the above query
SELECT usertype, ROUND(COUNT(DISTINCT CASE WHEN start_station_id = end_station_id THEN bikeid || CAST(starttime AS STRING) END) / 
        COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) * 100.0,1) pct_roundtrip
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
GROUP BY 1;

'''

pct_rndtrip_type_df = client.query(query).to_dataframe()
pct_rndtrip_type_df.style.format({'pct_roundtrip': '{:.1f}%'})

,pct_roundtrip
0,2%


,usertype,pct_roundtrip
0,Customer,5.9%
1,Subscriber,1.5%


### Q: What share of trips were round trips (same start and end station)? Does it differ by user type?
### A: 2% overall, 5.9% for Customers vs 1.5% for Subscribers. Citibike is designed for one-way trips. 
### Round trips are almost entirely a leisure behavior, which is why Customers are 4x more likely to do them.

In [37]:
# Among the top 20 start stations by volume, which station has the most lopsided rush-hour split (share of trips starting 7–10am vs 4–7pm on weekdays)? 
# Define “lopsided” however you want, but say what you mean. What might that tell you?
query = f'''
-- q4
WITH t20 AS
(
SELECT start_station_id
FROM
`bigquery-public-data.new_york_citibike.citibike_trips`

WHERE EXTRACT(YEAR FROM starttime) = 2017
GROUP BY 1
QUALIFY RANK() OVER( ORDER BY COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) DESC) <=20
),
am_pm AS
(
SELECT t.start_station_id, t.start_station_name,
  COUNT(DISTINCT
        CASE WHEN CAST(starttime AS TIME) BETWEEN '07:00:00' AND '10:00:00' THEN bikeid || CAST(starttime AS STRING) END
        ) / COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) am_rush_hr,
  COUNT(DISTINCT
        CASE WHEN CAST(starttime AS TIME) BETWEEN '16:00:00' AND '19:00:00' THEN bikeid || CAST(starttime AS STRING) END
        ) / COUNT(DISTINCT bikeid || CAST(starttime AS STRING)) pm_rush_hr
FROM
`bigquery-public-data.new_york_citibike.citibike_trips` t
JOIN 
t20
ON t.start_station_id = t20.start_station_id
WHERE EXTRACT(YEAR FROM starttime) = 2017
AND EXTRACT(DAYOFWEEK FROM starttime) BETWEEN 2 AND 6
GROUP BY 1,2
)
SELECT start_station_id, start_station_name, am_rush_hr, pm_rush_hr, ABS(am_rush_hr - pm_rush_hr) abs_diff

FROM am_pm
--ORDER BY 5 DESC
QUALIFY RANK() OVER(ORDER BY ABS(am_rush_hr - pm_rush_hr) DESC) = 1

'''

am_pm_df = client.query(query).to_dataframe()
am_pm_df.columns = ['start_station_id',	'start_station_name', 'am_rush_hr',	'pm_rush_hr', 'pctg_point_diff']
am_pm_df.style.format({'am_rush_hr': '{:.1%}', 'pm_rush_hr': '{:.1%}','pctg_point_diff': lambda x: f"{x * 100:.1f}"})

,start_station_id,start_station_name,am_rush_hr,pm_rush_hr,pctg_point_diff
0,281,Grand Army Plaza & Central Park S,5.3%,43.7%,38.4


### Q: Among the top 20 start stations by volume, which station has the most lopsided rush-hour split (share of trips starting 7–10am vs 4–7pm on weekdays)?
### Define “lopsided” however you want, but say what you mean. What might that tell you?
### A: Grand Army Plaza is overwhelmingly PM-heavy (43.7% PM vs 5.3% AM). To find lopsidedness, I'm using absolute difference between am and pm rush hours. This way I can easily sort by the magnitude of the difference. 

### The location at Central Park's south entrance near hotels makes it a magnet for after-work and evening leisure activity rather than morning commutes.
### Overall, the heavier rush hours tended to be after work rather than before. This makes sense because people usually try to get to work on a fixed schedule whereas after work, there is more flexibility for leisure activities.


# Question 2, Part B
## Option 1 - The two user types tell different stories. 

In [6]:
%%time

# Pulling in a random, deterministic sample to do EDA with - 5% sample
# This can run quickly on BQ but take some time to load into the df. A 10% sample took several minutes
query = f'''
SELECT *
FROM `bigquery-public-data.new_york_citibike.citibike_trips`
WHERE EXTRACT(YEAR FROM starttime) = 2017
AND MOD(ABS(FARM_FINGERPRINT(bikeid || CAST(starttime AS STRING))), 100) < 5 -- grabs a repeatable sample
'''
citibikes_df = client.query(query).to_dataframe()
citibikes_df

CPU times: total: 47.3 s
Wall time: 2min 36s


,tripduration,starttime,stoptime,start_station_id,start_station_name,start_station_latitude,start_station_longitude,end_station_id,end_station_name,end_station_latitude,end_station_longitude,bikeid,usertype,birth_year,gender,customer_plan
0,187,2017-10-30 08:36:16,2017-10-30 08:39:23,3458,W 55 St & 6 Ave,40.763094,-73.978350,520,W 52 St & 5 Ave,40.759923,-73.976485,31206,Subscriber,1963,male,
1,1700,2017-10-02 00:04:05,2017-10-02 00:32:25,3376,E 65 St & 2 Ave,40.764719,-73.962221,520,W 52 St & 5 Ave,40.759923,-73.976485,18312,Customer,<NA>,unknown,
2,232,2017-10-17 08:28:05,2017-10-17 08:31:57,3223,E 55 St & 3 Ave,40.758997,-73.968654,520,W 52 St & 5 Ave,40.759923,-73.976485,31291,Subscriber,1989,male,
3,1249,2017-05-02 17:29:06,2017-05-02 17:49:56,3238,E 80 St & 2 Ave,40.773914,-73.954395,520,W 52 St & 5 Ave,40.759923,-73.976485,28700,Subscriber,1994,male,
4,1029,2017-05-28 21:36:29,2017-05-28 21:53:38,3263,Cooper Square & E 7 St,40.729236,-73.990868,520,W 52 St & 5 Ave,40.759923,-73.976485,26361,Subscriber,1984,male,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
705171,899,2017-07-31 12:02:39,2017-07-31 12:17:39,3224,W 13 St & Hudson St,40.739974,-74.005139,358,Christopher St & Greenwich St,40.732916,-74.007114,30082,Customer,<NA>,unknown,
705172,589,2017-08-28 13:12:23,2017-08-28 13:22:12,3224,W 13 St & Hudson St,40.739974,-74.005139,358,Christopher St & Greenwich St,40.732916,-74.007114,30550,Customer,<NA>,unknown,
705173,2522,2017-09-13 12:12:39,2017-09-13 12:54:41,309,Murray St & West St,40.714979,-74.013012,358,Christopher St & Greenwich St,40.732916,-74.007114,26676,Customer,<NA>,unknown,
705174,332,2017-09-11 22:02:53,2017-09-11 22:08:25,3224,W 13 St & Hudson St,40.739974,-74.005139,358,Christopher St & Greenwich St,40.732916,-74.007114,25122,Customer,<NA>,unknown,


In [9]:
# look at the column names
citibikes_df.columns

Index(['tripduration', 'starttime', 'stoptime', 'start_station_id',
       'start_station_name', 'start_station_latitude',
       'start_station_longitude', 'end_station_id', 'end_station_name',
       'end_station_latitude', 'end_station_longitude', 'bikeid', 'usertype',
       'birth_year', 'gender', 'customer_plan'],
      dtype='object')

In [11]:
# Convert the numerical IDs to string
num_ids = ['start_station_id','end_station_id', 'bikeid']

citibikes_df[num_ids] = citibikes_df[num_ids].astype("string")
citibikes_df.head()

,tripduration,starttime,stoptime,start_station_id,start_station_name,start_station_latitude,start_station_longitude,end_station_id,end_station_name,end_station_latitude,end_station_longitude,bikeid,usertype,birth_year,gender,customer_plan
0,187,2017-10-30 08:36:16,2017-10-30 08:39:23,3458,W 55 St & 6 Ave,40.763094,-73.978350,520,W 52 St & 5 Ave,40.759923,-73.976485,31206,Subscriber,1963,male,
1,1700,2017-10-02 00:04:05,2017-10-02 00:32:25,3376,E 65 St & 2 Ave,40.764719,-73.962221,520,W 52 St & 5 Ave,40.759923,-73.976485,18312,Customer,<NA>,unknown,
2,232,2017-10-17 08:28:05,2017-10-17 08:31:57,3223,E 55 St & 3 Ave,40.758997,-73.968654,520,W 52 St & 5 Ave,40.759923,-73.976485,31291,Subscriber,1989,male,
3,1249,2017-05-02 17:29:06,2017-05-02 17:49:56,3238,E 80 St & 2 Ave,40.773914,-73.954395,520,W 52 St & 5 Ave,40.759923,-73.976485,28700,Subscriber,1994,male,
4,1029,2017-05-28 21:36:29,2017-05-28 21:53:38,3263,Cooper Square & E 7 St,40.729236,-73.990868,520,W 52 St & 5 Ave,40.759923,-73.976485,26361,Subscriber,1984,male,


In [14]:
# look at the numerical cols
citibikes_df.describe()

,tripduration,starttime,stoptime,start_station_latitude,start_station_longitude,end_station_latitude,end_station_longitude,birth_year
count,705176.0,705176,705176,705176.000000,705176.000000,705176.000000,705176.000000,628972.0
mean,1027.555156,2017-08-12 07:59:42.015207,2017-08-12 08:16:50.009175,40.736665,-73.984640,40.736280,-73.984713,1979.146301
min,61.0,2017-04-01 00:01:54,2017-04-01 00:11:40,40.646538,-74.034234,0.000000,-74.044725,1885.0
25%,379.0,2017-06-13 07:17:24,2017-06-13 07:32:00.750000,40.717746,-73.996674,40.717571,-73.997044,1971.0
50%,636.0,2017-08-13 22:32:48.500000,2017-08-13 22:46:26,40.737262,-73.987520,40.737050,-73.987538,1982.0
75%,1105.0,2017-10-10 16:21:15.250000,2017-10-10 16:37:19.750000,40.756014,-73.975748,40.755273,-73.976056,1988.0
max,8094296.0,2017-12-31 23:43:23,2018-01-08 14:43:30,45.505851,-73.569109,45.505851,0.000000,2001.0
std,18938.709932,NaN,NaN,0.029630,0.017478,0.056756,0.089826,11.779938


In [27]:
# look at the cat columns

num_cols = citibikes_df.describe().columns


# [col for col in citibikes_df.columns if col not in num_cols]
cat_df = citibikes_df.loc[:, ~citibikes_df.columns.isin(num_cols)]


cat_df.describe()

,start_station_id,start_station_name,end_station_id,end_station_name,bikeid,usertype,gender,customer_plan
count,705176,705176,705176,705176,705176,705176,705176,705176
unique,783,788,792,798,13672,2,3,1
top,519,Pershing Square North,519,Pershing Square North,27555,Subscriber,male,
freq,6823,6823,6724,6724,117,620997,464986,705176


In [35]:
# Check for nulls
citibikes_df.isnull().sum()

tripduration                   0
starttime                      0
stoptime                       0
start_station_id               0
start_station_name             0
start_station_latitude         0
start_station_longitude        0
end_station_id                 0
end_station_name               0
end_station_latitude           0
end_station_longitude          0
bikeid                         0
usertype                       0
birth_year                 76204
gender                         0
customer_plan                  0
dtype: int64

In [37]:
# Check for empty strings
cat_df.apply(lambda col: col.str.strip().eq('').sum())

start_station_id           0
start_station_name         0
end_station_id             0
end_station_name           0
bikeid                     0
usertype                   0
gender                     0
customer_plan         705176
dtype: int64

In [49]:
# Create a trip id
citibikes_df['trip_id'] = citibikes_df['bikeid'] + "_" + citibikes_df['starttime'].astype(str)
citibikes_df['trip_id'] 

0         31206_2017-10-30 08:36:16
1         18312_2017-10-02 00:04:05
2         31291_2017-10-17 08:28:05
3         28700_2017-05-02 17:29:06
4         26361_2017-05-28 21:36:29
                    ...            
705171    30082_2017-07-31 12:02:39
705172    30550_2017-08-28 13:12:23
705173    26676_2017-09-13 12:12:39
705174    25122_2017-09-11 22:02:53
705175    29172_2017-07-06 21:45:01
Name: trip_id, Length: 705176, dtype: string

In [50]:
# Check for dups
citibikes_df['trip_id'].duplicated().sum()

0

In [56]:
# Add some helper columns
citibikes_df['strt_hr'] = citibikes_df['starttime'].dt.hour
citibikes_df['end_hr'] = citibikes_df['stoptime'].dt.hour
citibikes_df['month'] = citibikes_df['starttime'].dt.month
citibikes_df['month_name'] = citibikes_df['starttime'].dt.month_name()
citibikes_df['weekday'] = citibikes_df['starttime'].dt.day_name()

In [57]:
# Check rides by month
citibikes_df.groupby(by='month')['trip_id'].count()

month
4     65738
5     76010
6     86301
7     86707
8     90623
9     94158
10    94956
11    66582
12    44101
Name: trip_id, dtype: int64

In [58]:
# investigating to make sure the original data also starts from april
# to make sure there wasn't an issue loading data to the DF and failing silently etc.

query = f'''
SELECT EXTRACT(YEAR FROM starttime) yr, 
       MIN(EXTRACT(MONTH FROM starttime)) first_month,
       MAX(EXTRACT(MONTH FROM starttime)) last_month,
       COUNT(*) trips
FROM `bigquery-public-data.new_york_citibike.citibike_trips`
GROUP BY 1
ORDER BY 1
'''
month_dist_df = client.query(query).to_dataframe()
month_dist_df

,yr,first_month,last_month,trips
0,<NA>,<NA>,<NA>,5828994
1,2013,7,12,5037185
2,2014,1,12,8081216
3,2015,1,12,9937969
4,2016,1,9,10262649
5,2017,4,12,14118669
6,2018,1,5,5671033


# 2017 is missing the first few months.
# Saving this as a branch and pivoting to 2015 so I can run a full year analysis.